## 1. Data Ingestion & Initial Profiling
In this step, we load the raw reviews dataset from the landing zone and reference the existing `silver_orders` table to establish our source of truth.

In [ ]:
# ── CELL 1: INITIALIZE SPARK SESSION ──────────────────────────────────────
import os
import pyspark
from pyspark.sql import SparkSession
from delta import configure_spark_with_delta_pip

# Build SparkSession locally to avoid Python version mismatch with external cluster
spark = SparkSession.builder \
    .appName("olist-notebook-analysis") \
    .master("local[*]") \
    .getOrCreate()

# Set log level to reduce noise inside the notebook
spark.sparkContext.setLogLevel("WARN")

print("Spark Session created successfully!")
print("Spark Master URI:", spark.sparkContext.master)

🎯 Spark Session created successfully!
Spark Master URI: local[*]


In [2]:
# ==========================================================================
# 1. LOAD DATASETS FROM MINIO CLOUD STORAGE
# ==========================================================================

# A. Load the order reviews dataset from the Bronze layer in MinIO
# Delta format handles quotes/escapes perfectly and preserves exact column types
df_reviews = (
    spark.read
    .format("delta")
    .load("s3a://bronze/csv/order_reviews/")
)

# Preview the clean bronze data inside the notebook
display(df_reviews.limit(10))


# B. Load the reference refined orders table from the Silver layer in MinIO
orders_silver = (
    spark.read
    .format("delta")
    .load("s3a://silver/refined/orders/")
)

DataFrame[review_id: string, order_id: string, review_score: int, review_comment_title: string, review_comment_message: string, review_creation_date: timestamp, review_answer_timestamp: timestamp, _ingested_at: timestamp, _source_file: string]

@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@

### Data Integrity: Pattern Filtering
* **Goal:** Enforce strict data quality by filtering out malformed records.
* **Logic:** Retains only rows where `review_id` matches the standard 32-character hexadecimal UUID pattern. This eliminates "shifted" data caused by CSV parsing errors, ensuring subsequent transformations run on clean, reliable records.

In [3]:
from pyspark.sql import functions as F
df_reviews = df_reviews.filter(F.col("review_id").rlike("^[a-fA-F0-9]{32}$"))

@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@

### Data Quality: Score Validation
* **Goal:** Ensure data usability for analysis.
* **Logic:** Filters out any records where `review_score` is `NULL`. Since the score is the primary metric for sentiment analysis, keeping only populated values guarantees the accuracy of performance KPIs and final labels.

In [4]:
from pyspark.sql import functions as F

df_reviews = df_reviews.filter(F.col("review_score").isNotNull())

In [5]:
# Calculate null counts for each column in the unique reviews dataset
null_counts = df_reviews.select([F.count(F.when(F.col(c).isNull(), c)).alias(c) for c in df_reviews.columns])

display(null_counts)

DataFrame[review_id: bigint, order_id: bigint, review_score: bigint, review_comment_title: bigint, review_comment_message: bigint, review_creation_date: bigint, review_answer_timestamp: bigint, _ingested_at: bigint, _source_file: bigint]

@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@

### Review Integrity Audit & Imputation
Detect missing or orphaned `order_id` values in customer reviews, log them for audit, and re-map them to a '-1' placeholder to ensure relational consistency.

In [6]:
from pyspark.sql import functions as F

# 1. Identify rows with missing order_id (NULLs)
null_id_df = df_reviews.filter(F.col("order_id").isNull()) \
    .withColumn("error_reason", F.lit("Critical: order_id is NULL")) \
    .withColumn("error_detected_at", F.current_timestamp())

# 2. Identify orphaned records (exist in reviews but missing from orders_silver)
orphaned_df = df_reviews.join(orders_silver, on="order_id", how="left_anti") \
    .filter(F.col("order_id").isNotNull()) \
    .withColumn("error_reason", F.lit("Orphaned: order_id not found in orders_silver")) \
    .withColumn("error_detected_at", F.current_timestamp())

# 3. Create 'reviews_errors_audit' as an independent DataFrame
# We combine both types of errors into this specific audit log
reviews_errors_audit = null_id_df.unionByName(orphaned_df, allowMissingColumns=True)

# 4. Clean df_reviews by filling NULLs and Orphans with -1
# We use the orphaned_df already identified above to find the list of IDs
orphaned_ids = [row.order_id for row in orphaned_df.select("order_id").distinct().collect()]

df_reviews = df_reviews.withColumn(
    "order_id",
    F.when(F.col("order_id").isNull(), F.lit("-1"))
     .when(F.col("order_id").isin(orphaned_ids), F.lit("-1"))
     .otherwise(F.col("order_id"))
)

# Reporting
print(f"Total error records logged in reviews_errors_audit: {reviews_errors_audit.count()}")
print("df_reviews has been updated: NULLs and Orphans replaced with -1.")

display(reviews_errors_audit)

Total error records logged in reviews_errors_audit: 0
df_reviews has been updated: NULLs and Orphans replaced with -1.


DataFrame[review_id: string, order_id: string, review_score: int, review_comment_title: string, review_comment_message: string, review_creation_date: timestamp, review_answer_timestamp: timestamp, _ingested_at: timestamp, _source_file: string, error_reason: string, error_detected_at: timestamp]

In [7]:
# Calculate null counts for each column in the unique reviews dataset
null_counts = df_reviews.select([F.count(F.when(F.col(c).isNull(), c)).alias(c) for c in df_reviews.columns])

display(null_counts)

DataFrame[review_id: bigint, order_id: bigint, review_score: bigint, review_comment_title: bigint, review_comment_message: bigint, review_creation_date: bigint, review_answer_timestamp: bigint, _ingested_at: bigint, _source_file: bigint]

@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@

### Data Quality Audit Report
Quantify the impact of data anomalies on the `reviews` dataset by calculating the percentage of null and orphaned records to monitor pipeline health and data integrity.

In [8]:
# --- 2. Stats & Reporting ---

# Calculate counts based on the error dataframes we identified earlier
total_reviews = df_reviews.count()
null_count = null_id_df.count()
orphaned_count = orphaned_df.count()
total_errors = null_count + orphaned_count

print(f"--- Data Quality Audit Report ---")
print(f"Total Reviews in Source: {total_reviews}")
print(f"Reviews with NULL IDs: {null_count} ({ (null_count/total_reviews)*100 :.2f}%)")
print(f"Orphaned Reviews: {orphaned_count} ({ (orphaned_count/total_reviews)*100 :.2f}%)")
print(f"Total Records Corrected (Flagged as -1): {total_errors} ({ (total_errors/total_reviews)*100 :.2f}%)")

--- Data Quality Audit Report ---
Total Reviews in Source: 99224
Reviews with NULL IDs: 0 (0.00%)
Orphaned Reviews: 0 (0.00%)
Total Records Corrected (Flagged as -1): 0 (0.00%)


In [9]:
# Check current schema before dropping columns
df_reviews.printSchema()

# Drop unstructured text columns to focus on structured metrics
df_reviews_cleaned = df_reviews.drop("review_comment_title", "review_comment_message")

root
 |-- review_id: string (nullable = true)
 |-- order_id: string (nullable = true)
 |-- review_score: integer (nullable = true)
 |-- review_comment_title: string (nullable = true)
 |-- review_comment_message: string (nullable = true)
 |-- review_creation_date: timestamp (nullable = true)
 |-- review_answer_timestamp: timestamp (nullable = true)
 |-- _ingested_at: timestamp (nullable = true)
 |-- _source_file: string (nullable = true)



@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@

### Schema Enforcement & Timestamp Standardization
Explicitly cast core review fields to their target types and parse date strings into standardized PySpark `TimestampType` objects to ensure operational consistency.

In [10]:
from pyspark.sql.types import StringType, IntegerType, TimestampType
df_reviews_cleaned = df_reviews_cleaned.select(
    F.col("review_id").cast(StringType()),
    F.col("order_id").cast(StringType()),
    F.col("review_score").cast(IntegerType()),
    F.to_timestamp(F.col("review_creation_date"), "yyyy-MM-dd HH:mm:ss").alias("review_creation_date"),
    F.to_timestamp(F.col("review_answer_timestamp"), "yyyy-MM-dd HH:mm:ss").alias("review_answer_timestamp")
)

In [11]:

print("Check NULLs after Casting:")
df_reviews_cleaned.filter(F.col("review_score").isNull()).show()

Check NULLs after Casting:
+---------+--------+------------+--------------------+-----------------------+
|review_id|order_id|review_score|review_creation_date|review_answer_timestamp|
+---------+--------+------------+--------------------+-----------------------+
+---------+--------+------------+--------------------+-----------------------+



In [12]:
# Calculate null counts for each column in the unique reviews dataset
null_counts = df_reviews_cleaned.select([F.count(F.when(F.col(c).isNull(), c)).alias(c) for c in df_reviews_cleaned.columns])

null_counts.show()

+---------+--------+------------+--------------------+-----------------------+
|review_id|order_id|review_score|review_creation_date|review_answer_timestamp|
+---------+--------+------------+--------------------+-----------------------+
|        0|       0|           0|                   0|                      0|
+---------+--------+------------+--------------------+-----------------------+



## 7. Descriptive Statistics & Duplicate Identification
We perform a statistical summary to understand the distribution of review scores and timestamps. Additionally, we calculate the total number of identical duplicate rows to ensure data uniqueness before further processing.

In [13]:
# 1. Calculate duplicates before removing them
total_count = df_reviews_cleaned.count()
distinct_df = df_reviews_cleaned.dropDuplicates()
distinct_count = distinct_df.count()

duplicate_count = total_count - distinct_count

# 2. Informative printing
print(f"--- Data Quality: Duplicates Report ---")
print(f"Total rows before cleaning: {total_count}")
print(f"Duplicate rows found: {duplicate_count}")
print(f"Total rows after cleaning: {distinct_count}")

# 3. Update the DataFrame to the cleaned version
df_reviews_cleaned = distinct_df

# 4. Generate descriptive statistics on the cleaned dataset
print("\n--- Descriptive Statistics (Cleaned Data) ---")
display(df_reviews_cleaned.describe())

--- Data Quality: Duplicates Report ---
Total rows before cleaning: 99224
Duplicate rows found: 0
Total rows after cleaning: 99224

--- Descriptive Statistics (Cleaned Data) ---


DataFrame[summary: string, review_id: string, order_id: string, review_score: string]

## 8. Identifying Logical Duplicates (Order-Level)
Beyond identical row duplicates, we need to investigate "Logical Duplicates" where multiple reviews are linked to a single `order_id`. This step identifies the scope of the issue before deep-dive analysis.

In [14]:
from pyspark.sql import functions as F

# Group by order_id to identify orders with multiple associated reviews
order_counts = df_reviews_cleaned.groupBy("order_id").count().filter("count > 1")

# Aggregate stats for logical duplicates
total_duplicates = order_counts.count()
total_extra_rows = order_counts.select(F.sum(F.col("count") - 1)).collect()[0][0]

print(f"Number of duplicate orders: {total_duplicates}")
print(f"Total Extra Rows to be resolved: {total_extra_rows}")

Number of duplicate orders: 547
Total Extra Rows to be resolved: 551


## 9. Deep Dive: Temporal Analysis of Duplicates
In this section, we analyze the time difference (in hours) between duplicate review entries. This helps distinguish between technical glitches (instant duplicates) and potential system-triggered reminders or customer updates.

In [15]:
from pyspark.sql import functions as F
from pyspark.sql import Window

# Join the duplicate IDs back to the main data to inspect their timestamps and scores
duplicate_check = df_reviews_cleaned.join(order_counts, "order_id", "inner") \
    .select("order_id", "review_answer_timestamp", "review_score") \
    .orderBy("order_id", "review_answer_timestamp")

# Define window specification to calculate time lag between reviews of the same order
w = Window.partitionBy("order_id").orderBy("review_answer_timestamp")

# Calculate the interval between consecutive reviews in hours
duplicate_check = duplicate_check.withColumn("prev_timestamp", F.lag("review_answer_timestamp").over(w)) \
    .withColumn("hours_between_reviews", 
                F.round((F.unix_timestamp("review_answer_timestamp") - F.unix_timestamp("prev_timestamp")) / 3600, 2)) \
    .filter(F.col("hours_between_reviews").isNotNull())

# Preview the time-gap analysis
duplicate_check.show(10)

+--------------------+-----------------------+------------+-------------------+---------------------+
|            order_id|review_answer_timestamp|review_score|     prev_timestamp|hours_between_reviews|
+--------------------+-----------------------+------------+-------------------+---------------------+
|0035246a40f520710...|    2017-08-30 01:59:12|           5|2017-08-29 21:45:57|                 4.22|
|013056cfe49763c6f...|    2018-03-05 17:02:00|           4|2018-02-23 12:12:30|               244.83|
|0176a6846bcb3b0d3...|    2018-01-02 10:54:47|           5|2018-01-02 10:54:06|                 0.01|
|02355020fd0a40a0d...|    2018-03-30 03:16:19|           1|2018-03-22 01:32:08|               193.74|
|029863af4b968de1e...|    2017-07-20 12:06:11|           4|2017-07-17 13:58:06|                70.13|
|02e0b68852217f571...|    2017-12-03 21:57:31|           4|2017-12-03 21:56:37|                 0.02|
|02e723e8edb4a123d...|    2017-09-05 12:12:51|           5|2017-09-02 12:13:03|   

## 10. Investigative Conclusion: Score Consistency Check
We examine whether the `review_score` changes between duplicate versions. If the scores are identical, it confirms a technical system duplication. If they differ, it indicates a customer updating their feedback. This evidence justifies our strategy of keeping the latest record.

In [16]:
# Check if the review score changed between the original and the duplicate version
check_behavior = duplicate_check.withColumn(
    "is_score_changed", 
    F.when(F.col("review_score") != F.lag("review_score").over(w), True).otherwise(False)
)

print("Analysis of Score Changes in Duplicates:")
check_behavior.groupBy("is_score_changed").count().show()

# Inspect samples where customers actually changed their scores
print("Sample of customers who changed their minds:")
check_behavior.filter(F.col("is_score_changed") == True)\
    .select("order_id", "hours_between_reviews", "review_score").show(5)

Analysis of Score Changes in Duplicates:
+----------------+-----+
|is_score_changed|count|
+----------------+-----+
|            true|    2|
|           false|  549|
+----------------+-----+

Sample of customers who changed their minds:
+--------------------+---------------------+------------+
|            order_id|hours_between_reviews|review_score|
+--------------------+---------------------+------------+
|03c939fd7fd3b38f8...|               214.01|           4|
|c88b1d1b157a9999c...|                 0.07|           5|
+--------------------+---------------------+------------+



## 11. Final Deduplication Strategy: Keeping the Latest Record
Based on our findings that the majority of duplicates are technical or rapid updates, we apply a Window Function to retain only the most recent review for each order. This ensures a 1:1 relationship between orders and reviews for our final Silver Layer.

In [17]:
from pyspark.sql import Window
from pyspark.sql import functions as F

# Apply Row Numbering to select the most recent record based on the answer timestamp
final_window = Window.partitionBy("order_id").orderBy(F.col("review_answer_timestamp").desc())

# Filter to keep only the latest version (row_num = 1)
df_reviews_unique = df_reviews_cleaned \
    .withColumn("row_num", F.row_number().over(final_window)) \
    .filter(F.col("row_num") == 1) \
    .drop("row_num")

print(f"Deduplication Complete! Final unique record count: {df_reviews_unique.count()}")

Deduplication Complete! Final unique record count: 98673


## 12. Null Values Profiling
After deduplication, we perform a comprehensive null check across all columns to identify any remaining gaps in our unique review records. This profiling step guides our final data quality filters.

In [18]:
# Calculate null counts for each column in the unique reviews dataset
null_counts = df_reviews_unique.select([F.count(F.when(F.col(c).isNull(), c)).alias(c) for c in df_reviews_unique.columns])

display(null_counts)

DataFrame[review_id: bigint, order_id: bigint, review_score: bigint, review_creation_date: bigint, review_answer_timestamp: bigint]

In [19]:
# Calculate counts and percentage for partial missing timestamps
total_count = df_reviews_unique.count()
null_creation_count = df_reviews_unique.filter(F.col("review_creation_date").isNull()).count()
null_percentage = (null_creation_count / total_count) * 100

print(f"Total Records: {total_count}")
print(f"Null Creation Date Records: {null_creation_count}")
print(f"Null Percentage: {null_percentage:.4f}%")

Total Records: 98673
Null Creation Date Records: 0
Null Percentage: 0.0000%


@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@

### Temporal Anomaly Repair
Detect missing date fields, log them for audit, and impute null `review_answer_timestamp` values using a calculated median response duration to maintain chronological integrity.

In [20]:
from pyspark.sql import functions as F

# 1. Identify rows with missing date information
temporal_errors_df = df_reviews_unique.filter(
    F.col("review_creation_date").isNull() | F.col("review_answer_timestamp").isNull()
) \
    .withColumn("error_reason", F.lit("Temporal Error: Missing date, imputed via Median")) \
    .withColumn("error_detected_at", F.current_timestamp())

# 2. Append records to the audit log
reviews_errors_audit = reviews_errors_audit.unionByName(temporal_errors_df, allowMissingColumns=True)

# 3. Calculate median response duration
median_stats = df_reviews_unique.filter(
    F.col("review_creation_date").isNotNull() & F.col("review_answer_timestamp").isNotNull()
).select(
    F.percentile_approx(F.unix_timestamp("review_answer_timestamp") - F.unix_timestamp("review_creation_date"), 0.5).alias("median_response_diff")
).collect()[0]

median_response_diff = median_stats["median_response_diff"]

# 4. Impute missing timestamps
df_reviews_unique = df_reviews_unique.withColumn(
    "review_answer_timestamp",
    F.when(
        F.col("review_answer_timestamp").isNull() & F.col("review_creation_date").isNotNull(),
        F.from_unixtime(F.unix_timestamp("review_creation_date") + median_response_diff)
    ).otherwise(F.col("review_answer_timestamp"))
)

print(f"Total temporal errors logged to reviews_errors_audit: {temporal_errors_df.count()}")
print("df_reviews_unique has been updated and cleaned (no flags added).")

Total temporal errors logged to reviews_errors_audit: 0
df_reviews_unique has been updated and cleaned (no flags added).


In [21]:
# Calculate null counts for each column in the unique reviews dataset
null_counts = df_reviews_unique.select([F.count(F.when(F.col(c).isNull(), c)).alias(c) for c in df_reviews_unique.columns])

display(null_counts)

DataFrame[review_id: bigint, order_id: bigint, review_score: bigint, review_creation_date: bigint, review_answer_timestamp: bigint]

@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@

### Cascading Temporal Imputation
Perform a two-stage imputation process by calculating purchase-to-creation and creation-to-answer medians, ensuring missing review timestamps are reconstructed logically based on the parent order's purchase history.

In [22]:
from pyspark.sql import functions as F

# 1. Identify records with missing temporal markers for audit logging
# Capture rows where both date fields are NULL before imputation
temporal_missing_df = df_reviews_unique.filter(
    F.col("review_creation_date").isNull() & 
    F.col("review_answer_timestamp").isNull()
).withColumn("error_reason", F.lit("Temporal Error: Both dates are NULL, imputed via Median")) \
 .withColumn("error_detected_at", F.current_timestamp())

# Append these errors to the 'reviews_errors_audit' DataFrame
reviews_errors_audit = reviews_errors_audit.unionByName(temporal_missing_df, allowMissingColumns=True)

# 2. Join with orders_silver to get purchase timestamp for calculation
df_enriched = df_reviews_unique.join(
    orders_silver.select("order_id", "order_purchase_timestamp"), 
    on="order_id", 
    how="left"
)

# 3. Calculate medians from records with complete data
stats = df_enriched.filter(
    F.col("review_creation_date").isNotNull() & 
    F.col("order_purchase_timestamp").isNotNull() & 
    F.col("review_answer_timestamp").isNotNull()
).select(
    F.percentile_approx(F.unix_timestamp("review_creation_date") - F.unix_timestamp("order_purchase_timestamp"), 0.5).alias("median_p2c"),
    F.percentile_approx(F.unix_timestamp("review_answer_timestamp") - F.unix_timestamp("review_creation_date"), 0.5).alias("median_c2a")
).collect()[0]

m_p2c = stats["median_p2c"] # Purchase to Creation median
m_c2a = stats["median_c2a"] # Creation to Answer median

# 4. Perform cascading imputation and define the final DataFrame
df_reviews_final = df_enriched.withColumn(
    # Stage 1: Impute Creation Date
    "review_creation_date",
    F.when(F.col("review_creation_date").isNull(), 
           F.from_unixtime(F.unix_timestamp("order_purchase_timestamp") + m_p2c))
     .otherwise(F.col("review_creation_date"))
).withColumn(
    # Stage 2: Impute Answer Timestamp based on the newly calculated Creation Date
    "review_answer_timestamp",
    F.when(F.col("review_answer_timestamp").isNull(), 
           F.from_unixtime(F.unix_timestamp("review_creation_date") + m_c2a))
     .otherwise(F.col("review_answer_timestamp"))
).drop("order_purchase_timestamp") # Clean up helper column

print(f"Total rows in df_reviews_final: {df_reviews_final.count()}")
print(f"Total temporal errors logged to reviews_errors_audit: {temporal_missing_df.count()}")

Total rows in df_reviews_final: 98673
Total temporal errors logged to reviews_errors_audit: 0


@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@

### Temporal Sequence Correction
Detect and resolve logical inconsistencies where review answer timestamps precede creation dates by performing an audit-logged swap, ensuring the chronological integrity of the review lifecycle.

In [23]:
from pyspark.sql import functions as F

# 1. Identify records where answer timestamp precedes creation date
# We create a dataframe of these anomalies to log them into the audit table
anomalies_df = df_reviews_final.filter(
    F.col("review_answer_timestamp") < F.col("review_creation_date")
).withColumn("error_reason", F.lit("Temporal Anomaly: Answer date precedes creation date, swapped")) \
 .withColumn("error_detected_at", F.current_timestamp())

# 2. Append anomalies to the 'reviews_errors_audit' DataFrame
reviews_errors_audit = reviews_errors_audit.unionByName(anomalies_df, allowMissingColumns=True)

# 3. Swap the values where the order is incorrect
# We use F.when to perform the swap only for the identified anomalous rows
df_reviews_final = df_reviews_final.withColumn(
    "temp_creation", F.col("review_creation_date")
).withColumn(
    "review_creation_date", 
    F.when(F.col("review_answer_timestamp") < F.col("review_creation_date"), F.col("review_answer_timestamp"))
     .otherwise(F.col("review_creation_date"))
).withColumn(
    "review_answer_timestamp", 
    F.when(F.col("review_answer_timestamp") < F.col("temp_creation"), F.col("temp_creation"))
     .otherwise(F.col("review_answer_timestamp"))
).drop("temp_creation")

# 4. Reporting
total_count = df_reviews_final.count()
anomalies_count = anomalies_df.count()

print(f"--- Temporal Anomaly Report ---")
print(f"Total Review Records: {total_count}")
print(f"Records with swapped dates (Corrected): {anomalies_count}")
print(f"Anomaly Percentage: {(anomalies_count / total_count) * 100:.2f}%")

--- Temporal Anomaly Report ---
Total Review Records: 98673
Records with swapped dates (Corrected): 0
Anomaly Percentage: 0.00%


## 19. Integrating Review and Order Timelines
To understand the context of each review, we perform an inner join with the `orders_silver` table. This allows us to compare review dates against key delivery milestones like carrier handover, actual delivery, and estimated delivery dates.

In [24]:
# Perform a quick join with the orders table to understand the temporal relationship
check_logic = df_reviews_final.join(orders_silver, "order_id", "inner") \
    .select(
        "review_creation_date", 
        "review_answer_timestamp",
        "order_delivered_carrier_date", 
        "order_delivered_customer_date",
        "order_estimated_delivery_date",
        "review_score"
    )

# Calculate the time lag in days between actual delivery and review creation
check_logic.withColumn("days_after_delivery", F.datediff("review_creation_date", "order_delivered_customer_date")).show(10)

+--------------------+-----------------------+----------------------------+-----------------------------+-----------------------------+------------+-------------------+
|review_creation_date|review_answer_timestamp|order_delivered_carrier_date|order_delivered_customer_date|order_estimated_delivery_date|review_score|days_after_delivery|
+--------------------+-----------------------+----------------------------+-----------------------------+-----------------------------+------------+-------------------+
| 2018-01-23 00:00:00|    2018-01-23 16:06:31|         2018-01-16 12:36:48|          2018-01-22 13:19:16|          2018-02-05 00:00:00|           5|                  1|
| 2017-03-02 00:00:00|    2017-03-03 10:54:59|         2017-02-16 09:46:09|          2017-03-01 16:42:31|          2017-03-17 00:00:00|           5|                  1|
| 2018-07-05 00:00:00|    2018-07-05 23:17:04|         2018-07-03 14:25:00|          2018-07-04 17:28:31|          2018-07-23 00:00:00|           4|       

## 20. Hypothesis Testing: Early Responses Analysis
We analyze "Early Responses" where a customer provides feedback before the actual delivery is recorded. We test whether these responses are linked to delayed shipments (past the estimated date) or represent true technical anomalies.

In [25]:
from pyspark.sql import functions as F

# 1. Filter for records where the answer preceded the actual delivery date
early_responses = check_logic.filter(
    F.col("review_answer_timestamp") < F.col("order_delivered_customer_date")
)

# 2. Analyze the relationship between early responses and estimated delivery dates
# Testing if the early response was triggered after the delivery became late (estimated date passed)
hypothesis_check = early_responses.withColumn(
    "is_after_estimated", 
    F.when(F.col("review_answer_timestamp") >= F.col("order_estimated_delivery_date"), "Yes (After Estimated)")
    .otherwise("No (Before Estimated - Anomaly)")
)

# 3. Aggregate results to observe patterns
print("Analysis of Early Responses (Answer < Actual Delivery):")
hypothesis_check.groupBy("is_after_estimated").agg(
    F.count("*").alias("count"),
    F.avg(F.datediff("review_answer_timestamp", "order_estimated_delivery_date")).alias("avg_days_diff")
).show()

# 4. Inspect samples of anomalies (Responses before both actual and estimated delivery)
print("Sample of Anomalies (Early Response AND Before Estimated):")
hypothesis_check.filter(F.col("is_after_estimated") == "No (Before Estimated - Anomaly)") \
    .select("review_answer_timestamp", "order_estimated_delivery_date", "order_delivered_customer_date") \
    .show(10)

Analysis of Early Responses (Answer < Actual Delivery):
+--------------------+-----+-------------------+
|  is_after_estimated|count|      avg_days_diff|
+--------------------+-----+-------------------+
|Yes (After Estima...| 4436|  3.767808836789901|
|No (Before Estima...|  242|-15.950413223140496|
+--------------------+-----+-------------------+

Sample of Anomalies (Early Response AND Before Estimated):
+-----------------------+-----------------------------+-----------------------------+
|review_answer_timestamp|order_estimated_delivery_date|order_delivered_customer_date|
+-----------------------+-----------------------------+-----------------------------+
|    2017-06-23 23:39:34|          2017-07-07 00:00:00|          2017-06-27 15:47:42|
|    2017-04-13 21:12:43|          2017-05-02 00:00:00|          2017-04-17 10:47:50|
|    2017-12-14 04:45:33|          2017-12-22 00:00:00|          2017-12-14 17:04:47|
|    2017-02-17 08:53:37|          2017-04-03 00:00:00|          2017-03-0

## 21. Exploratory Analysis: Reviews for Canceled Orders
In this step, we investigate whether canceled orders receive customer feedback. This exploration is essential to justify adding "Canceled" as a category in our `review_context` feature later, helping us understand if these records contribute to specific dissatisfaction patterns.

In [26]:
from pyspark.sql import functions as F

# 1. Join reviews with order status from the silver orders table to identify order states
check_cancel = df_reviews_final.join(
    orders_silver.select("order_id", "order_status", "order_delivered_customer_date"), 
    on="order_id", 
    how="inner"
)

# 2. Filter for canceled orders that actually have associated reviews
canceled_with_reviews = check_cancel.filter(F.col("order_status") == "canceled")

print(f"Number of reviews for cancelled orders: {canceled_with_reviews.count()}")

# 3. Analyze the score distribution to see how these customers rated their experience
canceled_with_reviews.groupBy("review_score").count().show()

Number of reviews for cancelled orders: 605
+------------+-----+
|review_score|count|
+------------+-----+
|           5|   66|
|           1|  421|
|           3|   48|
|           2|   44|
|           4|   26|
+------------+-----+



## 22. Validating the "Late Delivery Anger" Segment
We investigate a specific customer behavior: users who provide feedback after the estimated delivery date has passed but before the actual delivery occurs. This "Late Delivery Anger" segment is expected to have lower satisfaction scores.

In [27]:
from pyspark.sql import functions as F

# 1. Identify the "Late Delivery Anger" segment based on specific temporal conditions
anger_segment = check_logic.filter(
    (F.col("review_answer_timestamp") < F.col("order_delivered_customer_date")) & 
    (F.col("review_answer_timestamp") >= F.col("order_estimated_delivery_date"))
)

# 2. Calculate average score and star distribution (1-5)
anger_stats = anger_segment.select(
    F.avg("review_score").alias("Average_Score"),
    F.count("*").alias("Total_Count"),
    F.sum(F.when(F.col("review_score") <= 2, 1).otherwise(0)).alias("Low_Scores_Count"),
    F.sum(F.when(F.col("review_score") >= 4, 1).otherwise(0)).alias("High_Scores_Count")
)

# 3. Display the percentage of low ratings within this specific category
print("--- Validation: Late Delivery Anger Segment ---")
anger_stats.withColumn(
    "Low_Score_Percentage", 
    (F.col("Low_Scores_Count") / F.col("Total_Count")) * 100
).show()

# 4. Show detailed star rating distribution
anger_segment.groupBy("review_score").count().orderBy("review_score").show()

--- Validation: Late Delivery Anger Segment ---
+------------------+-----------+----------------+-----------------+--------------------+
|     Average_Score|Total_Count|Low_Scores_Count|High_Scores_Count|Low_Score_Percentage|
+------------------+-----------+----------------+-----------------+--------------------+
|1.6372858431018935|       4436|            3601|              455|   81.17673579801622|
+------------------+-----------+----------------+-----------------+--------------------+

+------------+-----+
|review_score|count|
+------------+-----+
|           1| 3165|
|           2|  436|
|           3|  380|
|           4|  189|
|           5|  266|
+------------+-----+



## 24. Sentiment Categorization: Sentiment Labeling
To simplify the analysis of customer feedback, we map the numerical `review_score` into a categorical `review_label`. This classification (Satisfied, Neutral, Unsatisfied) provides a high-level view of customer sentiment, making it more accessible for reporting and executive dashboards.

In [28]:
from pyspark.sql import functions as F

# Apply sentiment mapping based on the review score
# This creates a descriptive label: Satisfied (4-5), Neutral (3), or Unsatisfied (1-2)
df_reviews_final = df_reviews_final.withColumn(
    "review_label",
    F.when(F.col("review_score") >= 4, F.lit("Satisfied"))
    .when(F.col("review_score") == 3, F.lit("Neutral"))
    .otherwise(F.lit("Unsatisfied"))
)

## 25. Overall Review Score Distribution
Before deep-diving into specific segments, we need to understand the global distribution of customer satisfaction. [cite_start]This helps identify the baseline sentiment of the marketplace and informs us whether the data is skewed towards positive or negative feedback.

In [29]:
from pyspark.sql import functions as F

# 1. Calculate the distribution of each score (1-5)
score_distribution = df_reviews_final.groupBy("review_score").count().orderBy("review_score")

# 2. Calculate percentages for better business visibility
total_reviews = df_reviews_final.count()
score_distribution = score_distribution.withColumn(
    "percentage", 
    F.round((F.col("count") / total_reviews) * 100, 2)
)

print("Overall Review Score Distribution:")
score_distribution.show()

# 3. Quick Summary of Satisfaction Labels
df_reviews_final.groupBy("review_label").count().withColumn(
    "percentage", 
    F.round((F.col("count") / total_reviews) * 100, 2)
).show()

Overall Review Score Distribution:
+------------+-----+----------+
|review_score|count|percentage|
+------------+-----+----------+
|           1|11363|     11.52|
|           2| 3131|      3.17|
|           3| 8133|      8.24|
|           4|19038|     19.29|
|           5|57008|     57.77|
+------------+-----+----------+

+------------+-----+----------+
|review_label|count|percentage|
+------------+-----+----------+
|   Satisfied|76046|     77.07|
| Unsatisfied|14494|     14.69|
|     Neutral| 8133|      8.24|
+------------+-----+----------+



@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@

### Response Latency Analysis
Quantify customer service responsiveness by calculating temporal delays in days and hours, and categorize interactions to identify 'same-day' resolution performance.

In [30]:
from pyspark.sql import functions as F

# 1. Calculate the difference between creation and answer in days and hours
df_reviews_final = df_reviews_final.withColumn(
    "review_response_delay_days", 
    F.datediff("review_answer_timestamp", "review_creation_date")
).withColumn(
    "review_response_delay_hours",
    F.round((F.unix_timestamp("review_answer_timestamp") - F.unix_timestamp("review_creation_date")) / 3600, 2)
)

# 2. Add the flag: is_same_day_response
# True if difference in days is 0, else False
df_reviews_final = df_reviews_final.withColumn(
    "is_same_day_response",
    F.when(F.col("review_response_delay_days") == 0, F.lit(True))
     .otherwise(F.lit(False))
)

# 3. Statistical summary of the lag 
print("Response Time Statistics (in Days):")
df_reviews_final.select("review_response_delay_days").summary("mean", "min", "25%", "50%", "75%", "max").show()

# 4. Analyzing 'Quick Responders'
quick_responders = df_reviews_final.filter(F.col("is_same_day_response") == True).count()
total_reviews = df_reviews_final.count()

print(f"Number of records with same-day response: {quick_responders} ({ (quick_responders/total_reviews)*100 :.2f}%)")

Response Time Statistics (in Days):
+-------+--------------------------+
|summary|review_response_delay_days|
+-------+--------------------------+
|   mean|         2.585621193234218|
|    min|                         0|
|    25%|                         1|
|    50%|                         1|
|    75%|                         3|
|    max|                       518|
+-------+--------------------------+

Number of records with same-day response: 24236 (24.56%)


In [31]:
display(df_reviews_final.limit(10))

DataFrame[order_id: string, review_id: string, review_score: int, review_creation_date: string, review_answer_timestamp: string, review_label: string, review_response_delay_days: int, review_response_delay_hours: double, is_same_day_response: boolean]

In [32]:
df_reviews_final.printSchema()

root
 |-- order_id: string (nullable = true)
 |-- review_id: string (nullable = true)
 |-- review_score: integer (nullable = true)
 |-- review_creation_date: string (nullable = true)
 |-- review_answer_timestamp: string (nullable = true)
 |-- review_label: string (nullable = false)
 |-- review_response_delay_days: integer (nullable = true)
 |-- review_response_delay_hours: double (nullable = true)
 |-- is_same_day_response: boolean (nullable = false)



@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@

### Final Schema Standardization
Apply explicit data type casting and timestamp formatting to the finalized reviews dataset to ensure strict schema compliance for downstream silver-layer storage and analytical consumption.

In [33]:
from pyspark.sql.types import StringType, IntegerType, TimestampType

# Define the explicit mapping for casting
# Note: review_label and review_response_delay_days already match the types 
# in the schema you provided, so we focus on the raw fields and specific casts.
df_reviews_final = df_reviews_final \
    .withColumn("review_id", F.col("review_id").cast(StringType())) \
    .withColumn("order_id", F.col("order_id").cast(StringType())) \
    .withColumn("review_score", F.col("review_score").cast(IntegerType())) \
    .withColumn("review_creation_date", F.to_timestamp(F.col("review_creation_date"), "yyyy-MM-dd HH:mm:ss")) \
    .withColumn("review_answer_timestamp", F.to_timestamp(F.col("review_answer_timestamp"), "yyyy-MM-dd HH:mm:ss")) \
    .withColumn("review_label", F.col("review_label").cast(StringType())) \
    .withColumn("review_response_delay_days", F.col("review_response_delay_days").cast(IntegerType()))

# Print schema to confirm
df_reviews_final.printSchema()

root
 |-- order_id: string (nullable = true)
 |-- review_id: string (nullable = true)
 |-- review_score: integer (nullable = true)
 |-- review_creation_date: timestamp (nullable = true)
 |-- review_answer_timestamp: timestamp (nullable = true)
 |-- review_label: string (nullable = false)
 |-- review_response_delay_days: integer (nullable = true)
 |-- review_response_delay_hours: double (nullable = true)
 |-- is_same_day_response: boolean (nullable = false)



## 27. Data Persistence: Saving Silver Reviews Layer
We persist the fully enriched and cleaned reviews data into two formats: 
1. **Delta Table:** Inside the Metastore for optimized warehouse querying and time-travel capability.
2. **Single Parquet File:** Stored in the Lakehouse Files directory for portability and external tool integration.

In [34]:
# ==========================================================================
# FINAL PERSISTENCE: SAVING REFINED SILVER REVIEWS
# ==========================================================================

# Define the target absolute storage paths on MinIO
SILVER_REVIEWS_DELTA_PATH   = "s3a://silver/refined/reviews/"
SILVER_REVIEWS_PARQUET_PATH = "s3a://silver/refined/reviews_parquet/"

# FIX: Drop ambiguous metadata columns to prevent Delta duplicate columns error
df_reviews_final_cleaned = df_reviews_final.drop("_ingested_at", "_source_file")

# 1. Save as a Delta Table using direct MinIO S3A paths (FIXED: replaced saveAsTable)
df_reviews_final_cleaned.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .save(SILVER_REVIEWS_DELTA_PATH)

# Refresh the Delta cache for immediate query capability inside the cluster
spark.catalog.refreshByPath(SILVER_REVIEWS_DELTA_PATH)


# 2. Export as Parquet files to the Silver directory on MinIO (FIXED: updated local path to S3A)
# Coalesce(1) consolidates output into a single partition for easier downstream distribution
df_reviews_final_cleaned.coalesce(1).write \
    .mode("overwrite") \
    .parquet(SILVER_REVIEWS_PARQUET_PATH)

print("Success! Silver reviews table and parquet file are secured and safely saved to MinIO.")

Success! Silver reviews table and parquet file are secured and safely saved to MinIO.


In [35]:
# ==========================================================================
# AUDIT PERSISTENCE: SAVING REVIEWS QUALITY AUDIT LOGS
# ==========================================================================

# Define the absolute MinIO S3A storage path for the reviews audit logs
REVIEWS_AUDIT_LOG_PATH = "s3a://silver/qa_issues/silver_reviews_errors/"

# FIX: Drop ambiguous metadata columns from audit dataframe before persistence
reviews_errors_audit_cleaned = reviews_errors_audit.drop("_ingested_at", "_source_file")

print(f"Saving data quality audit logs to MinIO path: {REVIEWS_AUDIT_LOG_PATH}...")

# Save the audit dataframe using Delta format with append mode (FIXED: replaced saveAsTable)
reviews_errors_audit_cleaned.write \
    .format("delta") \
    .mode("append") \
    .option("overwriteSchema", "true") \
    .save(REVIEWS_AUDIT_LOG_PATH)

# Explicitly refresh the Delta cache for this path to ensure instant data governance visibility
spark.catalog.refreshByPath(REVIEWS_AUDIT_LOG_PATH)

print(f"Success! silver_reviews_errors_audit table saved successfully to MinIO path: {REVIEWS_AUDIT_LOG_PATH}")

Saving data quality audit logs to MinIO path: s3a://silver/qa_issues/silver_reviews_errors/...
Success! silver_reviews_errors_audit table saved successfully to MinIO path: s3a://silver/qa_issues/silver_reviews_errors/


## 28. Cross-Category Satisfaction Analysis (Threshold >= 50)
[cite_start]We analyze satisfaction across product categories with at least 50 reviews to ensure statistical significance, as recommended in the roadmap[cite: 271].

### Data Quality Note ("unknown"):
The **"unknown"** category shows 1,589 reviews. [cite_start]Since only 2 products lacked translation (manually fixed), the rest are original upstream Nulls[cite: 296]. [cite_start]We preserve these records to maintain accurate overall counts rather than deleting them[cite: 328].

In [36]:
from pyspark.sql import functions as F

# Load the refined reviews table from Silver layer
reviews_silver = (
    spark.read
    .format("delta")
    .load("s3a://silver/refined/reviews/")
)

# Load the refined order items intermediate bridge table from Silver layer
order_items_silver = (
    spark.read
    .format("delta")
    .load("s3a://silver/refined/order_items/")
)

# Load the refined products table from Silver layer
products_silver = (
    spark.read
    .format("delta")
    .load("s3a://silver/refined/products/")
)

# 2. Execute the full Join Path: reviews -> order_items -> products -> translation
df_reviews_categories = reviews_silver \
    .join(order_items_silver.select("order_id", "product_id"), "order_id", "inner") \
    .join(products_silver.select("product_id", "product_category_name"), "product_id", "inner")

# 3. Analyze satisfaction levels (review_label) across product categories
category_sentiment = df_reviews_categories.groupBy("product_category_name", "review_label").count()

# 4. Pivot the data to compare Satisfied, Neutral, and Unsatisfied counts side-by-side
category_pivot = category_sentiment.groupBy("product_category_name").pivot("review_label").sum("count").fillna(0)

# 5. Calculate Total Reviews per category and apply a filter threshold (e.g., minimum 50 reviews)
# This filters out the noise of small numbers that you correctly noticed!
category_analysis = category_pivot.withColumn(
    "Total_Reviews",
    (F.col("Satisfied") + F.col("Neutral") + F.col("Unsatisfied"))
).filter(F.col("Total_Reviews") >= 50)

# 6. Calculate 'Unsatisfied Rate' to reveal true high-risk business categories
category_analysis = category_analysis.withColumn(
    "Unsatisfied_Rate", 
    F.round((F.col("Unsatisfied") / F.col("Total_Reviews")) * 100, 2)
).orderBy(F.col("Unsatisfied_Rate").desc())

print("Top 10 Product Categories with Highest Dissatisfaction Rate (Minimum 50 Reviews):")
display(category_analysis.limit(10))

Top 10 Product Categories with Highest Dissatisfaction Rate (Minimum 50 Reviews):


DataFrame[product_category_name: string, Neutral: bigint, Satisfied: bigint, Unsatisfied: bigint, Total_Reviews: bigint, Unsatisfied_Rate: double]

## 29. Seller Segment Analysis: Order Volume vs. Customer Satisfaction
Following the roadmap (Step 4 & 5), we group sellers into Small (<50 orders), Medium (50-500), and Large (>500) to see if rapid merchant growth correlates with higher dissatisfaction rates.

In [37]:
from pyspark.sql import functions as F


# 2. Join enriched reviews with seller IDs
df_reviews_sellers = reviews_silver.join(
    order_items_silver.select("order_id", "seller_id"), 
    on="order_id", 
    how="inner"
)

# 3. Calculate total reviews per seller to define segments
seller_volumes = df_reviews_sellers.groupBy("seller_id").count().withColumnRenamed("count", "seller_order_count")

# 4. Create Seller Segments based on the roadmap criteria
seller_segments = seller_volumes.withColumn(
    "seller_segment",
    F.when(F.col("seller_order_count") < 50, F.lit("Small Seller"))
    .when((F.col("seller_order_count") >= 50) & (F.col("seller_order_count") <= 500), F.lit("Medium Seller"))
    .otherwise(F.lit("Large Seller"))
)

# 5. Join segments back and pivot by satisfaction label
df_segmented_reviews = df_reviews_sellers.join(seller_segments.select("seller_id", "seller_segment"), "seller_id", "inner")
segment_sentiment = df_segmented_reviews.groupBy("seller_segment", "review_label").count()

# 6. Pivot for final business presentation
segment_pivot = segment_sentiment.groupBy("seller_segment").pivot("review_label").sum("count").fillna(0)

# 7. Calculate Unsatisfied Rate per segment
final_seller_analysis = segment_pivot.withColumn(
    "Total_Reviews",
    (F.col("Satisfied") + F.col("Neutral") + F.col("Unsatisfied"))
).withColumn(
    "Unsatisfied_Rate",
    F.round((F.col("Unsatisfied") / F.col("Total_Reviews")) * 100, 2)
).orderBy("Total_Reviews")

print("Seller Segment Performance Analysis:")
display(final_seller_analysis)

Seller Segment Performance Analysis:


DataFrame[seller_segment: string, Neutral: bigint, Satisfied: bigint, Unsatisfied: bigint, Total_Reviews: bigint, Unsatisfied_Rate: double]